# What makes a Roblox game popular?
**Research question: Which game characteristics are most strongly associated with popularity on Roblox?**

Course: AI-Powered Data Analysis using Python & Agile (Sarah Bouraga), EM Normandie
Group: Aliénor TRUONG-MINH-CHIEU, Léonel DA SILVA BANDEIRA, Rim [NOM], Manel [NOM]

## 1. Introduction
**Problem statement.** Roblox hosts millions of games, but only a few become hits. Game creators and studios need to know where to put their effort: which genre to choose, whether to keep updating the game, how to present it.

**Business question.** Which game characteristics (genre, age, update frequency, server size, description, creator status, icon colours) are most strongly associated with popularity?

**Why it matters.** A creator who knows which characteristics go together with success can make better decisions before and after launch.

**Popularity measure.** We use the total number of visits of each game. We check below that it moves together with the two other popularity signals: players online now and favourites.

**Data.** Ready-made Roblox datasets (Kaggle, GitHub) were too small (less than 10,000 games), old, or missing the variables we needed. We therefore built our own dataset in two steps, in a separate collection notebook run once on 15 November 2026:
1. a public archive of Roblox game icons (Frrazers) gave us a list of 28,530 game IDs;
2. for each ID we asked the official Roblox games API (games.roblox.com/v1/games) for the game information, and measured simple colour statistics of its icon.

After keeping only games with complete information we have **24,006 games**. The table was frozen as a CSV file and is loaded below from its URL, so this notebook does not depend on the Roblox API.

## 2. Packages required
We import pandas (tables), numpy (maths), matplotlib and seaborn (charts) and scikit-learn (a simple prediction model).

In [ ]:
# Author: Aliénor
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression

## 3. Data Cleaning and Preparation
We load the data from its URL and look at its size.

In [ ]:
# Author: Aliénor
url = "https://raw.githubusercontent.com/alienooooor/roblox-cc-python/main/roblox_games.csv"
df = pd.read_csv(url)
df.shape

We look at the first rows to understand the columns.

In [ ]:
# Author: Aliénor
df.head()

We check the type of each column.

In [ ]:
# Author: Aliénor
df.info()

We check for missing values.

In [ ]:
# Author: Aliénor
df.isna().sum()

We check that each game appears only once.

In [ ]:
# Author: Léonel
df["universe_id"].duplicated().sum()

The dates are stored as text: we convert them into real dates.

In [ ]:
# Author: Léonel
df["created"] = pd.to_datetime(df["created"], format="ISO8601")
df["updated"] = pd.to_datetime(df["updated"], format="ISO8601")

We remove the few impossible rows: games with 0 visits (they would break the log scale used later) and games with a negative number of favourites (data error).

In [ ]:
# Author: Léonel
df = df[(df["visits"] > 0) & (df["favorites"] >= 0)]
df.shape

**Result:** no missing values, no duplicates, dates converted, 12 impossible rows removed: we keep 23,994 games.

## 4. Exploratory Data Analysis
Summary statistics of the main numeric columns.

In [ ]:
# Author: Léonel
df[["visits", "playing", "favorites", "max_players", "description_length"]].describe()

The mean number of visits is much larger than the median: a few huge hits pull the average up. That is why we use **medians** and a **log scale** below.

Distribution of visits (log scale).

In [ ]:
# Author: Léonel
sns.histplot(df["visits"], log_scale=True)
plt.title("Distribution of total visits per game")
plt.xlabel("Total visits (log scale)")
plt.ylabel("Number of games")
plt.show()

**Interpretation:** most games have between 500,000 and 20 million visits (median ≈ 3.9 million). Only 1% of games pass 780 million visits: success on Roblox is very concentrated.

Is visits a good popularity measure? We compare it with players online now and favourites (Spearman correlation, robust to extreme values).

In [ ]:
# Author: Léonel
df[["visits", "playing", "favorites"]].corr(method="spearman")

**Interpretation:** visits are strongly linked to favourites (0.83) and to current players (0.67): the three signals tell the same story, so visits is a fair popularity measure.

Median visits by genre (only genres with at least 100 games, so that medians are reliable).

In [ ]:
# Author: Léonel
counts = df["genre"].value_counts()
big_genres = counts[counts >= 100].index
df[df["genre"].isin(big_genres)].groupby("genre")["visits"].median().sort_values().plot(kind="barh")
plt.title("Median visits by genre")
plt.xlabel("Median total visits")
plt.show()

**Interpretation:** Sports, Fighting, FPS, Horror and Military games have the highest median visits (around 20 million or more). Simulation, the most common genre (about 1 game in 3), only reaches 2.3 million, and Shopping is last. The most crowded genre is not the most successful one.

Median visits for verified vs non-verified creators.

In [ ]:
# Author: Léonel
df.groupby("creator_verified")["visits"].median()

**Interpretation:** games from verified creators have a median of about 26.6 million visits, against 2.0 million for the others: more than 13 times more.

## 5. Feature Engineering
We create 5 new features from the existing columns.

**Feature 1: popularity level.** A function puts each game in a class: Low (< 1 million visits), Medium (1 to 10 million), High (> 10 million).

In [ ]:
# Author: Rim
def popularity_level(visits):
    if visits < 1_000_000:
        return "Low"
    elif visits < 10_000_000:
        return "Medium"
    else:
        return "High"

df["popularity_level"] = df["visits"].apply(popularity_level)
df["popularity_level"].value_counts()

**Features 2, 3 and 4: age of the game, days since last update, recently updated (updated in the last 30 days).** The reference date is the day the data was collected (latest update in the data).

In [ ]:
# Author: Rim
today = df["updated"].max()
df["age_years"] = (today - df["created"]).dt.days / 365
df["days_since_update"] = (today - df["updated"]).dt.days
df["recently_updated"] = df["days_since_update"] <= 30

**Feature 5: log of visits.** Visits go from a few hundred to 90 billion; the log (base 10) puts them on a readable scale (6 = 1 million, 7 = 10 million).

In [ ]:
# Author: Rim
df["log_visits"] = np.log10(df["visits"])

Age of the game by popularity level.

In [ ]:
# Author: Rim
sns.boxplot(data=df, x="popularity_level", y="age_years", order=["Low", "Medium", "High"])
plt.title("Age of the game by popularity level")
plt.xlabel("Popularity level")
plt.ylabel("Age of the game (years)")
plt.show()

**Interpretation:** popular games are clearly older. Visits accumulate over time, so age is one of the strongest characteristics.

Median visits for games updated in the last 30 days vs the others.

In [ ]:
# Author: Rim
df.groupby("recently_updated")["visits"].median()

**Interpretation:** games updated in the last 30 days have a median of about 6.5 million visits, against 2.9 million for the others: active games are more popular.

Which characteristics are most linked to popularity? Correlation of each characteristic with log visits.

In [ ]:
# Author: Rim
cols = ["age_years", "days_since_update", "max_players", "description_length",
        "icon_brightness", "icon_saturation", "icon_colorfulness", "creator_verified"]
corr = df[cols + ["log_visits"]].corr()["log_visits"].drop("log_visits")
corr.sort_values().plot(kind="barh")
plt.title("Correlation with popularity (log visits)")
plt.xlabel("Correlation coefficient")
plt.show()

**Interpretation:** the age of the game (0.44) and having a verified creator (0.41) are by far the most linked to popularity. Server size, description length and icon colours have almost no linear link (|r| < 0.06).

**Simple prediction.** A linear regression predicts log visits from the three strongest characteristics. The R² tells which share of the differences in popularity they explain.

In [ ]:
# Author: Manel
X = df[["age_years", "creator_verified", "recently_updated"]]
y = df["log_visits"]
model = LinearRegression().fit(X, y)
print("R2:", round(model.score(X, y), 2))
print(dict(zip(X.columns, model.coef_.round(2))))

**Interpretation:** these 3 characteristics alone explain about one third of the differences in popularity (R² ≈ 0.34). On the log scale (+1 = 10 times more visits): a verified creator adds about +1.07 (≈ 12 times more visits), each extra year of age adds about +0.21 (≈ 1.6 times more), and a recent update adds about +0.36 (≈ 2.3 times more), all other things being equal.

## 6. Summary
**Problem and approach.** We asked which characteristics are most strongly associated with popularity on Roblox. We built a dataset of 23,994 games (public archive + official Roblox API), cleaned it, explored it, created 5 new features and fitted a simple linear regression.

**Insights.**
1. **Age** is the strongest factor: older games have accumulated far more visits.
2. **Verified creators** have about 13 times more median visits than the others.
3. **Updating the game** matters: games updated in the last 30 days have about twice the median visits.
4. **Genre** matters: Sports, Fighting, FPS, Horror and Military do best; Simulation is the most crowded but under average; Shopping does worst.
5. **Server size, description length and icon colours** have almost no link with popularity.

**Managerial implications.** A creator should (i) avoid over-crowded genres like Simulation, or stand out within them, (ii) keep updating the game regularly, (iii) build trust and reach (a verified studio or group), and (iv) be patient: popularity is built over years. Spending time on icon colours or on a longer description is not where the gain is.

**Limitations.** (i) Our games come from a public archive, not a random sample of all Roblox games. (ii) Correlation is not causation: for example verified creators are often already big studios. (iii) Visits accumulate over time, which favours old games. (iv) The data is a snapshot taken on 15 November 2026. (v) The icon measures are simple colour statistics, not a judgement of design quality. (vi) The regression is basic: it explains only about one third of the differences, so many factors (gameplay quality, marketing, YouTube videos) are not in our data.

## Declaration of Generative AI and AI-assisted technologies in the writing process
During the preparation of this work the author(s) used ChatGPT (OpenAI) and Claude (Anthropic) in order to find the tools and functions needed to collect the data from the Roblox API, and to improve the readability of the code and of the text. After using these tools, the author(s) reviewed and edited the content as needed and take(s) full responsibility for the content of the publication.

Prompts used (summary):
- ChatGPT (data collection notebook): we asked for a Google Colab notebook that collects at least 10,000 Roblox games from public sources and the official Roblox API, and merges them into one table. [to replace by the exact prompts if found]
- Claude: we asked to summarise the project instructions, to explain the collection notebook in simple words, to check that the collected table had at least 10,000 complete games, to suggest short, course-level pandas/seaborn code for each section, and to explain each line of code so that every member can defend it.

## References
- Roblox games API: https://games.roblox.com/v1/games
- Frrazers, Roblox icons dataset: https://tools.frrazers.com/dataset/
- Our frozen dataset (CSV): https://raw.githubusercontent.com/alienooooor/roblox-cc-python/main/roblox_games.csv
- pandas documentation: https://pandas.pydata.org/docs/
- seaborn documentation: https://seaborn.pydata.org/
- scikit-learn, LinearRegression: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html
- Course material: S. Bouraga, AI-Powered Data Analysis using Python & Agile, EM Normandie, 2026.